# October Task #3: Target and Mask Table + Scorer Spec

**Goal:** one chunk x 10-category table of targets (`positive` / `negative` / `masked`) for the **train** and **validation** splits, plus a draft scorer spec. Amie's training harness (Task #4) and the evaluator (Task #8) read the output.

**How it works:** it reuses the *same* paragraph-based chunking and the *same* target rules that Task 10 validated for training data (`task10_chunking_val.ipynb`), and runs them for the validation contracts too. Then it cross-checks everything:
- the train side must reproduce Task 10's committed numbers exactly,
- the validation chunk IDs are compared with the validation chunks in `task12b_outputs` (the files named `test_*`, which Amie loads as validation),
- every table is re-derived a second, independent way (span-first instead of chunk-first).

**Rules this notebook follows**
- Train and validation only. The official test set is never touched (it is reserved for the locked run, Task #11).
- Nothing is written to the repo. Outputs go to `october_task3_outputs/` in Colab; the last cell zips them so you can download and commit them yourself.

**Runtime:** CPU is fine (no GPU needed). Run all cells top to bottom.

## 1. Setup

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/Accenture-1O-contract-review-challenge/main/"
)
OUTPUT_DIR = Path("october_task3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

checks = []

def add_check(section, check, expected, actual, passed=None, note=""):
    if passed is None:
        passed = (actual == expected)
    checks.append({"section": section, "check": check, "expected": expected,
                   "actual": actual, "passed": bool(passed), "note": note})

def show_checks(section):
    df = pd.DataFrame(checks)
    df = df[df["section"] == section].reset_index(drop=True)
    display(df)
    n_fail = int((~df["passed"]).sum())
    print(f"{section}: {len(df) - n_fail}/{len(df)} checks passed" + ("" if n_fail == 0 else f"  <-- {n_fail} FAILED, investigate before using the output"))

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

## 2. Frozen configuration

The chunking config below is the one Task 10 froze (`notebooks/task10_outputs/chunking_config_frozen.json`) and that the training chunks used. The category order is fixed and matches `categories` in `october_task1_outputs/task1_frozen_model_specification.json`.

In [ ]:
import requests
chunking_config = json.loads(requests.get(BASE_URL + "notebooks/task10_outputs/chunking_config_frozen.json", timeout=60).text)
print(json.dumps(chunking_config, indent=2))

EXPECTED = {"method": "paragraph_based", "tokenizer_name": "distilroberta-base",
            "chunk_size": 512, "long_paragraph_overlap": 64}

SELECTED_CATEGORIES = [
    "governing_law", "renewal_term", "revenue_profit_sharing", "cap_on_liability",
    "uncapped_liability", "termination_for_convenience", "anti_assignment",
    "audit_rights", "license_grant", "exclusivity",
]
N_CATS = len(SELECTED_CATEGORIES)

S = "2_config"
for key, val in EXPECTED.items():
    add_check(S, f"config_{key}", val, chunking_config.get(key))
add_check(S, "categories_count_unique", 10, len(set(SELECTED_CATEGORIES)))

spec = json.loads(requests.get(BASE_URL + "notebooks/october_task1_outputs/task1_frozen_model_specification.json", timeout=60).text)
add_check(S, "category_order_matches_task1_spec", SELECTED_CATEGORIES, list(spec["categories"]))
show_checks(S)

## 3. Load data and the frozen split

In [ ]:
documents = pd.read_parquet(BASE_URL + "notebooks/split_data/documents.parquet")
annotation_sets = pd.read_parquet(BASE_URL + "notebooks/split_data/annotation_sets.parquet")
spans = pd.read_parquet(BASE_URL + "notebooks/split_data/spans.parquet")
categories = pd.read_parquet(BASE_URL + "notebooks/split_data/categories.parquet")
split_map = pd.read_parquet(BASE_URL + "notebooks/frozen-split_data/frozen-split.parquet")

print("frozen split columns:", split_map.columns.tolist())
print(split_map["split"].value_counts(dropna=False))
print("groups per split:\n", split_map.groupby("split")["context_group_id"].nunique())

split_labels = sorted(split_map["split"].dropna().unique())
TRAIN_LABEL = "train"
val_candidates = [s for s in split_labels if str(s).lower().startswith("val")]
assert TRAIN_LABEL in split_labels, f"no 'train' label in {split_labels}"
assert len(val_candidates) == 1, f"could not identify the validation label in {split_labels}"
VAL_LABEL = val_candidates[0]
print("Using labels -> train:", TRAIN_LABEL, "| validation:", VAL_LABEL)

docs_all = documents.merge(split_map, on="context_group_id", how="left")

def docs_for(split_label):
    return (docs_all[docs_all["split"] == split_label]
            .sort_values("contract_id").reset_index(drop=True))

train_docs = docs_for(TRAIN_LABEL)
val_docs = docs_for(VAL_LABEL)

S = "3_data"
add_check(S, "train_documents", 325, len(train_docs), note="Task 9/10 reference")
add_check(S, "validation_context_groups", 83, int(val_docs["context_group_id"].nunique()), note="Aisha's email / Beth: 83 validation groups")
n_unlabeled = int(docs_all["split"].isna().sum())
add_check(S, "documents_without_split_label_excluded", n_unlabeled, n_unlabeled, passed=True,
          note="informational: contracts with no train/validation label are never used here")
add_check(S, "no_contract_in_both_splits", 0,
          len(set(train_docs["contract_id"]) & set(val_docs["contract_id"])))
add_check(S, "no_group_in_both_splits", 0,
          len(set(train_docs["context_group_id"]) & set(val_docs["context_group_id"])))
show_checks(S)
print("validation contracts:", len(val_docs))

## 4. Paragraph-based chunker (copied from Task 10)

Same logic Task 10 validated. Each chunk is a character range `[char_start, char_end)` of a contract.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(chunking_config["tokenizer_name"], use_fast=True)
num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

def make_chunk_record(method, contract_id, group_id, chunk_number, char_start, char_end, n_content_tokens):
    return {
        "method": method,
        "contract_id": contract_id,
        "context_group_id": group_id,
        "chunk_id": f"{contract_id}__chunk_{chunk_number:04d}",
        "chunk_number": chunk_number,
        "char_start": int(char_start),
        "char_end": int(char_end),
        "num_tokens": int(n_content_tokens + num_special_tokens),
    }

def create_paragraph_chunks(documents_df, chunk_size, long_paragraph_overlap, method_name):
    # Pack whole paragraphs into a chunk until the token limit; a single paragraph longer than
    # the limit is split into token windows with a small overlap.
    max_content = chunk_size - num_special_tokens
    records = []
    for _, row in documents_df.iterrows():
        text = str(row["context"])
        contract_records = []

        def emit(char_start, char_end, n_content_tokens):
            contract_records.append(make_chunk_record(
                method_name, row["contract_id"], row["context_group_id"],
                len(contract_records), char_start, char_end, n_content_tokens))

        paragraphs = list(re.finditer(r"\S(?:.*?)(?=\n\s*\n|\Z)", text, flags=re.DOTALL))
        current_start, current_end, current_tokens = None, None, 0

        for para in paragraphs:
            encoded = tokenizer(para.group(0), add_special_tokens=False, truncation=False, return_offsets_mapping=True)
            para_offsets = encoded["offset_mapping"]
            para_tokens = len(encoded["input_ids"])

            if para_tokens > max_content:
                if current_start is not None:
                    emit(current_start, current_end, current_tokens)
                    current_start, current_end, current_tokens = None, None, 0
                step = max_content - long_paragraph_overlap
                for start in range(0, para_tokens, step):
                    end = min(start + max_content, para_tokens)
                    piece = para_offsets[start:end]
                    if not piece:
                        continue
                    emit(para.start() + piece[0][0], para.start() + piece[-1][1], end - start)
                    if end == para_tokens:
                        break
                continue

            if current_tokens + para_tokens <= max_content:
                if current_start is None:
                    current_start = para.start()
                current_end = para.end()
                current_tokens += para_tokens
            else:
                emit(current_start, current_end, current_tokens)
                current_start, current_end, current_tokens = para.start(), para.end(), para_tokens

        if current_start is not None:
            emit(current_start, current_end, current_tokens)
        records.extend(contract_records)
    return pd.DataFrame(records)

def generate_chunks(documents_df):
    chunks_df = create_paragraph_chunks(
        documents_df, chunking_config["chunk_size"],
        chunking_config["long_paragraph_overlap"], chunking_config["method"])
    chunks_df = chunks_df.sort_values(["contract_id", "chunk_number"]).reset_index(drop=True)
    text_of = dict(zip(documents_df["contract_id"], documents_df["context"]))
    chunks_df["chunk_text"] = [text_of[c][s:e] for c, s, e in
                               zip(chunks_df["contract_id"], chunks_df["char_start"], chunks_df["char_end"])]
    return chunks_df

## 5. Generate chunks for train and validation

In [ ]:
train_chunks = generate_chunks(train_docs)
val_chunks = generate_chunks(val_docs)

for name, ch in [("train", train_chunks), ("validation", val_chunks)]:
    print(f"{name}: {len(ch)} chunks across {ch['contract_id'].nunique()} contracts; "
          f"max tokens {ch['num_tokens'].max()}")

S = "5_chunks"
add_check(S, "train_chunks_vs_task10", 10841, len(train_chunks), note="paragraph_based reference from Task 9/10")
add_check(S, "every_train_contract_has_chunks", len(train_docs), train_chunks["contract_id"].nunique())
add_check(S, "every_validation_contract_has_chunks", len(val_docs), val_chunks["contract_id"].nunique())
add_check(S, "empty_chunk_texts", 0, int((pd.concat([train_chunks, val_chunks])["chunk_text"].str.strip() == "").sum()))
add_check(S, "max_tokens_le_512", True, bool(pd.concat([train_chunks, val_chunks])["num_tokens"].max() <= 512))
show_checks(S)

## 6. Cross-check against the chunks Amie trains on

Amie's harness loads `task12b_outputs/train_chunks.parquet` and `task12b_outputs/test_chunks.parquet` (the latter is the **validation** set, named with the prefix "test" by Task 12). The chunk IDs and text produced here must match them, otherwise the masks will not line up with her rows.

In [ ]:
S = "6_match_task12b"
try:
    amie_train = pd.read_parquet(BASE_URL + "notebooks/task12b_outputs/train_chunks.parquet")
    amie_val = pd.read_parquet(BASE_URL + "notebooks/task12b_outputs/test_chunks.parquet")
    print("task12b train columns:", amie_train.columns.tolist())
    print("task12b 'test' (validation) columns:", amie_val.columns.tolist())

    for name, mine, theirs in [("train", train_chunks, amie_train), ("validation", val_chunks, amie_val)]:
        my_ids, their_ids = set(mine["chunk_id"]), set(theirs["chunk_id"])
        add_check(S, f"{name}_chunk_id_sets_equal", True, my_ids == their_ids,
                  note=f"mine {len(my_ids)} vs task12b {len(their_ids)}; only-mine {len(my_ids - their_ids)}, only-theirs {len(their_ids - my_ids)}")
        add_check(S, f"{name}_chunk_counts_equal", len(theirs), len(mine))
        if "text" in theirs.columns:
            joined = mine.merge(theirs[["chunk_id", "text"]], on="chunk_id", how="inner")
            n_same = int((joined["chunk_text"] == joined["text"]).sum())
            n_same_stripped = int((joined["chunk_text"].str.strip() == joined["text"].str.strip()).sum())
            add_check(S, f"{name}_chunk_text_identical", len(joined), n_same,
                      passed=(n_same_stripped == len(joined)),
                      note=f"exact {n_same}/{len(joined)}, equal after strip {n_same_stripped}/{len(joined)}")
        # order should also match, because Amie builds labels by chunk order
        if my_ids == their_ids:
            add_check(S, f"{name}_chunk_order_identical", True,
                      mine["chunk_id"].tolist() == theirs["chunk_id"].tolist())
except Exception as exc:
    print("Could not compare with task12b outputs:", repr(exc))
    add_check(S, "task12b_comparison_ran", True, False, note=repr(exc))
show_checks(S)

## 7. Targets and masks

The rules are the Task 10 rules, unchanged. For each chunk and category:

1. Category not annotated in the contract -> `negative`.
2. Annotated, but no span has usable offsets -> `masked` (location unknown).
3. Otherwise compare the chunk's character range with each annotated span:
   - the chunk fully **contains** a span -> `positive`
   - otherwise, it only **partly overlaps** a span (a boundary split) -> `masked`
   - otherwise -> `negative`

`label` is 1 for positive, 0 for negative, and missing for masked. For training, `mask` = 0 for masked pairs and 1 otherwise, and masked labels are stored as 0 (they never count).

In [ ]:
def classify_chunk(chunk_start, chunk_end, is_annotated, span_ranges):
    if not is_annotated:
        return "negative", 0, 0
    if not span_ranges:
        return "masked", 0, 0
    n_contained = n_partial = 0
    for span_start, span_end in span_ranges:
        if chunk_start <= span_start and chunk_end >= span_end:
            n_contained += 1
        elif chunk_start < span_end and chunk_end > span_start:
            n_partial += 1
    if n_contained:
        return "positive", n_contained, n_partial
    if n_partial:
        return "masked", n_contained, n_partial
    return "negative", n_contained, n_partial


def build_targets(split_label, docs_df, chunks_df):
    # Returns (chunk_category_rows, evaluation_spans, positive_annotation_sets, unlocated_pairs)
    contract_ids = set(docs_df["contract_id"])
    split_sets = annotation_sets[annotation_sets["contract_id"].isin(contract_ids)]
    split_spans = spans[spans["annotation_set_id"].isin(split_sets["annotation_set_id"])]
    selected_sets = split_sets[split_sets["category_id"].isin(SELECTED_CATEGORIES)]
    positive_sets = selected_sets[selected_sets["is_impossible"] == False]

    positive_spans_raw = split_spans[split_spans["annotation_set_id"].isin(positive_sets["annotation_set_id"])]
    eval_spans = positive_spans_raw.dropna(subset=["answer_start", "answer_end"]).copy()
    eval_spans[["answer_start", "answer_end"]] = eval_spans[["answer_start", "answer_end"]].astype(int)
    eval_spans = eval_spans.merge(
        positive_sets[["annotation_set_id", "contract_id", "category_id"]],
        on="annotation_set_id", how="left").reset_index(drop=True)

    existing_pairs = set(zip(selected_sets["contract_id"], selected_sets["category_id"]))
    missing_pairs = [(c, k) for c in contract_ids for k in SELECTED_CATEGORIES if (c, k) not in existing_pairs]

    rows = (chunks_df[["chunk_id", "contract_id", "context_group_id", "chunk_number"]]
            .loc[chunks_df.index.repeat(N_CATS)].reset_index(drop=True))
    rows["category_id"] = np.tile(SELECTED_CATEGORIES, len(chunks_df))

    annotated_pairs = set(zip(positive_sets["contract_id"], positive_sets["category_id"]))
    span_ranges_by_pair = {
        pair: list(zip(g["answer_start"], g["answer_end"]))
        for pair, g in eval_spans.groupby(["contract_id", "category_id"])
    }
    unlocated = sorted(annotated_pairs - set(span_ranges_by_pair))

    results = [
        classify_chunk(cs, ce, (contract, cat) in annotated_pairs, span_ranges_by_pair.get((contract, cat), []))
        for contract, cat, cs, ce in zip(
            rows["contract_id"], rows["category_id"],
            np.repeat(chunks_df["char_start"].to_numpy(), N_CATS),
            np.repeat(chunks_df["char_end"].to_numpy(), N_CATS))
    ]
    rows[["target", "n_contained_spans", "n_partial_spans"]] = pd.DataFrame(results)
    rows["label"] = rows["target"].map({"positive": 1, "negative": 0}).astype("Int8")
    rows["is_masked"] = rows["target"] == "masked"
    rows["annotation_present"] = [p in annotated_pairs for p in zip(rows["contract_id"], rows["category_id"])]
    rows.insert(0, "split", split_label)
    return rows, eval_spans, positive_sets, unlocated, missing_pairs, len(positive_spans_raw)


train_rows, train_spans_eval, train_pos_sets, train_unlocated, train_missing, train_raw_n = build_targets(TRAIN_LABEL, train_docs, train_chunks)
val_rows, val_spans_eval, val_pos_sets, val_unlocated, val_missing, val_raw_n = build_targets(VAL_LABEL, val_docs, val_chunks)

for name, rows in [("train", train_rows), ("validation", val_rows)]:
    print(f"\n{name}: {len(rows)} rows")
    print(rows["target"].value_counts().to_string())

## 8. Validate: train must reproduce Task 10, both splits must be internally consistent

In [ ]:
S = "8_validate_targets"

# ---- train regression against Task 10's committed summary ----
ref = pd.read_csv(BASE_URL + "notebooks/task10_outputs/target_summary_by_category.csv").set_index("category_id")
mine = (train_rows.pivot_table(index="category_id", columns="target", values="chunk_id", aggfunc="count", fill_value=0)
        .reindex(SELECTED_CATEGORIES).reindex(columns=["positive", "masked", "negative"], fill_value=0))
add_check(S, "train_rows_vs_task10", 108410, len(train_rows))
add_check(S, "train_positive_masked_negative_equal_task10_per_category", True,
          bool((mine.loc[ref.index, ["positive", "masked", "negative"]].to_numpy()
                == ref[["positive", "masked", "negative"]].to_numpy()).all()))

# ---- structure checks, both splits ----
for name, rows, chunks_df, missing in [("train", train_rows, train_chunks, train_missing),
                                       ("validation", val_rows, val_chunks, val_missing)]:
    add_check(S, f"{name}_rows_equal_chunks_x_10", len(chunks_df) * N_CATS, len(rows))
    add_check(S, f"{name}_every_chunk_has_all_10_categories", len(chunks_df),
              int((rows.groupby("chunk_id")["category_id"].nunique() == N_CATS).sum()))
    add_check(S, f"{name}_duplicate_chunk_category_rows", 0, int(rows.duplicated(["chunk_id", "category_id"]).sum()))
    add_check(S, f"{name}_invalid_targets", 0, int((~rows["target"].isin(["positive", "negative", "masked"])).sum()))
    add_check(S, f"{name}_contract_category_pairs_missing_annotation_set", 0, len(missing))
    add_check(S, f"{name}_positive_or_masked_without_annotation", 0,
              int(((rows["target"] != "negative") & ~rows["annotation_present"]).sum()))
    add_check(S, f"{name}_split_label_in_rows", True, bool((rows["split"] == (TRAIN_LABEL if name == 'train' else VAL_LABEL)).all()))

# ---- independent span-first re-derivation ----
def audit_spans(chunks_df, spans_df):
    chunks_of = {c: g for c, g in chunks_df.groupby("contract_id")}
    exp_pos, exp_mask = set(), set()
    n_split = 0
    for sp in spans_df.itertuples(index=False):
        touched = False
        for ch in chunks_of.get(sp.contract_id, pd.DataFrame()).itertuples(index=False):
            contains = ch.char_start <= sp.answer_start and ch.char_end >= sp.answer_end
            touches = ch.char_start < sp.answer_end and ch.char_end > sp.answer_start
            if contains:
                exp_pos.add((ch.chunk_id, sp.category_id))
            elif touches:
                exp_mask.add((ch.chunk_id, sp.category_id))
    return exp_pos, exp_mask

for name, rows, chunks_df, eval_spans, unlocated, raw_n in [
        ("train", train_rows, train_chunks, train_spans_eval, train_unlocated, train_raw_n),
        ("validation", val_rows, val_chunks, val_spans_eval, val_unlocated, val_raw_n)]:
    exp_pos, exp_mask = audit_spans(chunks_df, eval_spans)
    for contract, cat in unlocated:
        exp_mask |= {(cid, cat) for cid in chunks_df.loc[chunks_df["contract_id"] == contract, "chunk_id"]}
    exp_mask -= exp_pos
    def pairs_with_target(target):
        sub = rows[rows["target"] == target]
        return set(zip(sub["chunk_id"], sub["category_id"]))
    got_pos = pairs_with_target("positive")
    got_mask = pairs_with_target("masked")
    add_check(S, f"{name}_positive_rows_differing_from_independent_recompute", 0, len(got_pos ^ exp_pos))
    add_check(S, f"{name}_masked_rows_differing_from_independent_recompute", 0, len(got_mask ^ exp_mask))
    add_check(S, f"{name}_positive_spans_dropped_for_missing_offsets", 0, raw_n - len(eval_spans),
              note="spans with no usable offsets are handled as masked, not dropped silently")

# ---- no positive annotation is lost (every positive annotation set has >=1 positive chunk) ----
for name, rows, pos_sets in [("train", train_rows, train_pos_sets), ("validation", val_rows, val_pos_sets)]:
    pos_counts = rows[rows["target"] == "positive"].groupby(["contract_id", "category_id"]).size()
    no_pos = [pair for pair in zip(pos_sets["contract_id"], pos_sets["category_id"]) if int(pos_counts.get(pair, 0)) == 0]
    # Documented exceptions: annotation has no fully-contained chunk (boundary split or no usable offsets),
    # so it is masked. The real failure would be an annotation that is neither positive nor masked.
    masked_counts = rows[rows["target"] == "masked"].groupby(["contract_id", "category_id"]).size()
    silently_lost = [pair for pair in no_pos if int(masked_counts.get(pair, 0)) == 0]
    add_check(S, f"{name}_annotation_sets_neither_positive_nor_masked", 0, len(silently_lost),
              note=("" if not silently_lost else f"examples: {silently_lost[:5]}"))
    add_check(S, f"{name}_documented_exceptions_masked_no_positive_chunk", len(no_pos), len(no_pos), passed=True,
              note=f"informational: {len(no_pos)} annotation sets have no fully-contained chunk (all masked). Task 10 train reference: 20")
    pd.DataFrame(no_pos, columns=["contract_id", "category_id"]).assign(split=name).to_csv(
        OUTPUT_DIR / f"task3_{name}_masked_only_annotations.csv", index=False)
show_checks(S)

## 9. Target counts by split and category

In [ ]:
def summarize(rows):
    t = (rows.pivot_table(index="category_id", columns="target", values="chunk_id", aggfunc="count", fill_value=0)
         .reindex(SELECTED_CATEGORIES).reindex(columns=["positive", "masked", "negative"], fill_value=0))
    t["total"] = t.sum(axis=1)
    t["masked_pct"] = (t["masked"] / t["total"] * 100).round(2)
    return t

summary = pd.concat({TRAIN_LABEL: summarize(train_rows), VAL_LABEL: summarize(val_rows)}, names=["split", "category_id"])
display(summary)

low = summary[(summary["positive"] < 10)]
if len(low):
    print("\nNOTE: categories with fewer than 10 positive chunks in a split (metrics will be noisy / may be undefined):")
    display(low)

## 10. Build the files Amie loads

One row per chunk with `labels` and `mask`, both lists of 10 ints in the fixed category order. Masked pairs get `label = 0` and `mask = 0`, so a mask-aware loss ignores them.

In [ ]:
def to_wide(rows, chunks_df):
    lab = rows.pivot(index="chunk_id", columns="category_id", values="label")[SELECTED_CATEGORIES]
    msk = rows.pivot(index="chunk_id", columns="category_id", values="is_masked")[SELECTED_CATEGORIES]
    order = chunks_df["chunk_id"].tolist()
    lab, msk = lab.loc[order], msk.loc[order]
    mask_int = (~msk.astype(bool)).astype(int)                 # 1 = counts, 0 = masked
    labels_int = lab.fillna(0).astype(int)                     # masked -> 0 (ignored via mask)
    wide = pd.DataFrame({
        "chunk_id": order,
        "labels": labels_int.to_numpy().tolist(),
        "mask": mask_int.to_numpy().tolist(),
    })
    return wide

train_wide = to_wide(train_rows, train_chunks)
val_wide = to_wide(val_rows, val_chunks)

S = "10_wide"
for name, wide, rows, chunks_df in [("train", train_wide, train_rows, train_chunks), ("validation", val_wide, val_rows, val_chunks)]:
    add_check(S, f"{name}_wide_rows_equal_chunks", len(chunks_df), len(wide))
    add_check(S, f"{name}_all_vectors_length_10", True,
              bool(wide["labels"].map(len).eq(N_CATS).all() and wide["mask"].map(len).eq(N_CATS).all()))
    add_check(S, f"{name}_masked_pairs_in_mask_vectors", int(rows["is_masked"].sum()),
              int(sum(N_CATS - sum(m) for m in wide["mask"])))
    add_check(S, f"{name}_positive_labels_in_vectors", int((rows["target"] == "positive").sum()),
              int(sum(sum(l) for l in wide["labels"])))
    add_check(S, f"{name}_no_positive_label_is_masked", 0,
              int(sum(1 for l, m in zip(wide["labels"], wide["mask"]) if any(li == 1 and mi == 0 for li, mi in zip(l, m)))))
show_checks(S)
display(val_wide.head(3))

## 11. Draft scorer spec

Draft for the team to review against the October milestone wording and Beth's feedback. The statements about what the model and evaluator do come from Beth's Oct 22 email and the frozen spec. The threshold wording follows the frozen model spec; check it matches Task #1.

In [ ]:
scorer_spec = """# Scorer spec (DRAFT for team review) - October Task #3

## Unit of scoring
One **chunk-category pair**. The chunk set is the approved paragraph-based chunking (config hash in
`task10_outputs/chunking_config_frozen.json`). Each chunk has exactly 10 pairs, one per category, in this fixed order:
governing_law, renewal_term, revenue_profit_sharing, cap_on_liability, uncapped_liability,
termination_for_convenience, anti_assignment, audit_rights, license_grant, exclusivity.

## One score, one detection
- The model and each baseline emit **exactly one score per pair** (10 scores per chunk, no more, no fewer).
- A pair is **detected** when its score is >= that category's frozen threshold; otherwise it is not detected.
- Thresholds are chosen on the validation split only, then frozen before the locked run. No threshold or
  weight changes after the locked run.

## Targets
- `positive`: the chunk fully contains at least one annotated span for the category.
- `negative`: the category is not annotated in the contract, or the annotation does not touch the chunk.
- `masked`: the annotation has no usable offsets, or the chunk only partly overlaps a span (boundary split).

## What is scored
- Metrics (per-category precision, recall, F1; macro-F1 headline) are computed on **unmasked pairs only**.
- Masked pairs are excluded from the loss, from class-weight counts, from threshold selection, and from metrics.
- No post-processing between the score and the detection (no top-k, no cross-category rules).

## Splits
- Training: class weights (pos_weight = negatives / positives, unmasked pairs, per category) come from the train split only.
- Validation: model selection and threshold selection.
- Test: touched once, in the locked run (Task #11).

## Open items (confirm with the team / Beth)
- Confirm paragraph-based chunking is the approved set (the Task 9 settings file lists a 512/128 sliding window).
- Confirm masked pairs are excluded from the metrics for all three systems, including the baselines.
"""
(OUTPUT_DIR / "task3_scorer_spec_DRAFT.md").write_text(scorer_spec)
print(scorer_spec)

## 12. Save outputs and final check

In [ ]:
all_rows = pd.concat([train_rows, val_rows], ignore_index=True)
all_rows.drop(columns=["method"], errors="ignore").to_csv(OUTPUT_DIR / "task3_chunk_category_targets.csv", index=False)
train_wide.to_parquet(OUTPUT_DIR / "task3_labels_mask_train.parquet", index=False)
val_wide.to_parquet(OUTPUT_DIR / "task3_labels_mask_validation.parquet", index=False)
summary.to_csv(OUTPUT_DIR / "task3_target_summary_by_split.csv")
pd.DataFrame(checks).to_csv(OUTPUT_DIR / "task3_validation_checks.csv", index=False)

failed = [c for c in checks if not c["passed"]]
print(f"{len(checks) - len(failed)}/{len(checks)} checks passed")
for c in failed:
    print("FAILED:", c["section"], c["check"], "| expected:", c["expected"], "| actual:", c["actual"], "|", c["note"])
print("\nFiles written:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p, f"({p.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Zip for download (Colab). Commit these from your own clone, on your own branch.
import shutil
shutil.make_archive("october_task3_outputs", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download("october_task3_outputs.zip")
except Exception as exc:
    print("Not running in Colab, zip is at october_task3_outputs.zip", repr(exc))

## How Amie plugs this in

Replace the `TEMPORARY` cell in `task4_training_harness.ipynb` (the one that sets the placeholder mask) with:

```python
import pandas as pd

def load_labels_mask(path):
    df = pd.read_parquet(path).set_index("chunk_id")
    return df

t3_train = load_labels_mask("october_task3_outputs/task3_labels_mask_train.parquet")
t3_val = load_labels_mask("october_task3_outputs/task3_labels_mask_validation.parquet")

def build_dataset(chunks, t3):
    dataset = chunks[["chunk_id", "text"]].copy()
    dataset["labels"] = [list(map(int, x)) for x in t3.loc[dataset["chunk_id"], "labels"]]
    dataset["mask"] = [list(map(int, x)) for x in t3.loc[dataset["chunk_id"], "mask"]]
    return dataset

train_dataset = build_dataset(train_chunks, t3_train)
val_dataset = build_dataset(val_chunks, t3_val)   # val_chunks = task12b "test_*" files = validation
```

The category order in `labels` and `mask` is the same as `FINAL_IDS` in her notebook.